# Experimento 2 — LightGBM com balanceamento utilizando 4 variáveis

O segundo experimento mantém o mesmo algoritmo e condições do
Experimento 1, introduzindo o parâmetro `class_weight="balanced"`
como única modificação.

O `class_weight="balanced"` ajusta automaticamente os pesos de cada
classe durante o treinamento, sendo inversamente proporcional à sua
frequência no dataset. Classes com menos amostras recebem peso maior,
forçando o modelo a penalizá-las mais quando erra — o que aumenta a
atenção do algoritmo às classes minoritárias `Crítica` e `Atenção`.

As variáveis utilizadas como entrada foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Country`
- `Waterbody Type`

Enquanto isso, a variável alvo (`y`) utilizada para previsão foi:

- `conama_status`

O dataset foi dividido em:

- 80% para treino
- 20% para teste

A separação foi realizada utilizando `train_test_split` com `random_state=42`
e `stratify=y`, mantendo a proporção original das classes nos conjuntos de
treino e teste.

O objetivo deste experimento é isolar o efeito do balanceamento —
qualquer diferença nos resultados em relação ao Experimento 1 pode ser
atribuída exclusivamente ao `class_weight="balanced"`.

Após o treinamento, o modelo foi avaliado utilizando:

- Accuracy
- Precision
- Recall
- F1-score
- Matriz de confusão

## Preparação do ambiente

In [ ]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")
# esperado: (141399, 22)

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from lightgbm import LGBMClassifier

In [ ]:
# DEFINIÇÃO DE X E Y
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type"
]]

y = df["conama_status"]

In [ ]:
# parâmetros - divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 4)
Teste: (28280, 4)


In [ ]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(class_weight='balanced', n_jobs=-1,
                                random_state=42, verbose=-1))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi
realizada a avaliação das métricas no conjunto de treino, permitindo
comparar o comportamento entre treino e teste e identificar possíveis
variações no overfitting em relação aos experimentos anteriores.


In [ ]:
# Métricas de treino
y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.6126380183700351
Train Precision:
0.7249531986332473
Train Recall:
0.6126380183700351
Train F1:
0.6527494661284595
Train Confusion Matrix:
[[ 4444   518  2116   482]
 [ 5657  6587  2613  6821]
 [  199    20   862    25]
 [ 8120 14298  2949 57408]]


In [ ]:
# Métricas de teste
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.59992927864215

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.22      0.53      0.32      1890
         Boa       0.29      0.29      0.29      5419
     Crítica       0.06      0.51      0.11       277
   Excelente       0.88      0.69      0.77     20694

    accuracy                           0.60     28280
   macro avg       0.36      0.50      0.37     28280
weighted avg       0.72      0.60      0.64     28280


Confusion Matrix:
[[ 1010   155   592   133]
 [ 1385  1546   709  1779]
 [  104    24   140     9]
 [ 1993  3691   740 14270]]


## Resultados Obtidos — Experimento 2

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.599
```

Enquanto isso, a acurácia no conjunto de treino ficou em torno de:

```python
0.612
```

### Comparação com Experimento 1

| Métrica             | Exp 1 — LGBM s/ bal | Exp 2 — LGBM c/ bal |
|---------------------|----------------------|----------------------|
| Accuracy treino     | 0.75                 | 0.612                |
| Accuracy teste      | 0.745                | 0.599                |
| Precision Atenção   | 0.42                 | 0.22                 |
| Precision Boa       | 0.39                 | 0.29                 |
| Precision Crítica   | 0.17                 | 0.06                 |
| Precision Excelente | 0.79                 | 0.88                 |
| Recall Atenção      | 0.21                 | 0.53                 |
| Recall Boa          | 0.16                 | 0.29                 |
| Recall Crítica      | 0.01                 | 0.51                 |
| Recall Excelente    | 0.96                 | 0.69                 |
| F1 Atenção          | 0.28                 | 0.32                 |
| F1 Boa              | 0.23                 | 0.29                 |
| F1 Crítica          | 0.02                 | 0.11                 |
| F1 Excelente        | 0.87                 | 0.77                 |
| Overfitting         | 0.005                | 0.013                |

### Comparação entre algoritmos com balanceamento
*Random Forest (Experimento 2) vs LightGBM (Experimento 2)*

| Métrica             | RF c/ bal | LGBM c/ bal |
|---------------------|-----------|-------------|
| Accuracy treino     | 0.838     | 0.612       |
| Accuracy teste      | 0.651     | 0.599       |
| Overfitting         | 0.187     | 0.013       |
| Precision Atenção   | 0.22      | 0.22        |
| Precision Boa       | 0.29      | 0.29        |
| Precision Crítica   | 0.05      | 0.06        |
| Precision Excelente | 0.82      | 0.88        |
| Recall Atenção      | 0.44      | 0.53        |
| Recall Boa          | 0.23      | 0.29        |
| Recall Crítica      | 0.08      | 0.51        |
| Recall Excelente    | 0.79      | 0.69        |
| F1 Atenção          | 0.30      | 0.32        |
| F1 Boa              | 0.25      | 0.29        |
| F1 Crítica          | 0.07      | 0.11        |
| F1 Excelente        | 0.81      | 0.77        |

### Conclusão

A introdução do `class_weight="balanced"` provocou a inversão mais
expressiva entre precision e recall de toda a sequência experimental,
com impacto direto no comportamento do modelo frente às classes
minoritárias.

Em `Crítica`, o movimento foi o mais significativo: precision caiu de
0.17 para 0.06, enquanto recall saltou de 0.01 para 0.51. Isso significa
que o modelo passou a prever `Crítica` com muito mais frequência —
identificando corretamente mais da metade das amostras reais dessa
classe — porém ao custo de um aumento expressivo de falsos positivos.
No contexto de qualidade hídrica, esse trade-off é favorável: deixar
de detectar uma condição crítica real tem consequências ambientais
mais graves do que classificar erroneamente uma amostra como crítica.
O F1 subiu de 0.02 para 0.11, refletindo o ganho líquido dessa
redistribuição.

`Atenção` seguiu o mesmo padrão — precision caiu de 0.42 para 0.22
e recall subiu de 0.21 para 0.53. O modelo dobrou sua capacidade de
detecção dessa classe, ainda que com menor precisão nas previsões.
O F1 avançou de 0.28 para 0.32.

`Boa` apresentou comportamento mais equilibrado entre as duas métricas:
precision caiu de 0.39 para 0.29 e recall subiu de 0.16 para 0.29,
resultando em F1 de 0.29 ante 0.23 — o ganho mais balanceado entre
as classes minoritárias.

Em `Excelente`, o efeito foi inverso: recall caiu de 0.96 para 0.69,
indicando que o modelo deixou de classificar automaticamente a maioria
das amostras como `Excelente`. A precision subiu de 0.79 para 0.88 —
quando o modelo previu `Excelente`, acertou com maior frequência —
mas o volume de previsões corretas reduziu, resultando em queda do F1
de 0.87 para 0.77.

Comparando os dois algoritmos com balanceamento, o LightGBM demonstrou
vantagem consistente nas classes minoritárias. O recall de `Crítica`
foi o diferencial mais expressivo: 0.51 contra 0.08 do Random Forest —
o LightGBM identificou mais de seis vezes mais amostras críticas reais.
`Atenção` e `Boa` também tiveram recall superior no LightGBM, com
ganhos em F1 em todas as classes minoritárias. A única vantagem do
Random Forest foi em `Excelente` (F1 0.81 vs 0.77) e acurácia geral
(0.651 vs 0.599), ambas sustentadas pelo maior favorecimento da classe
majoritária — comportamento evidenciado pelo overfitting de 0.187,
muito superior ao 0.013 do LightGBM.

O overfitting do LightGBM subiu levemente de 0.005 para 0.013 com a
adição do balanceamento, permanecendo muito abaixo do observado no
Random Forest e confirmando que o `class_weight="balanced"` não
comprometeu a capacidade de generalização do modelo.

In [ ]:
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="weighted")

resultados = {
    "experimento": "exp_02_lightgbm_com_balanceamento",
    "algoritmo": "LightGBM",
    "balanceamento": "class_weight=balanced",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas salvas.")

Métricas salvas.
